In [ ]:
# Program to understand the BGP announcements specific for potential scrubbers
# Note: All the helper methods (e.g. detect_ip_version) are at the bottom of the program file
# Step # 1
# Find customers (ASN and prefixes) of DDoS scrubber for a day
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

scrubber_asn = "57724"
stream = pybgpstream.BGPStream(
    from_time="2024-07-07 00:00:01 CET",
    until_time="2024-07-07 23:59:00 CET",
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
        filter = "path _"+scrubber_asn+"_" #Look for all the prefixes originated by AS200020
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'announcements')


pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
#         print("## elem %s" %elem)
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]
        orig = as_path.split()[-1]
        second_as = as_path.split()[-2]

        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # 2. Prefix length should be /24 or /48
        # See all the prefixes that has AS200020 as a second last hop and only IPv4
        if second_as == scrubber_asn and detect_ip_version(pfx) == 'IPv4':
#             print("Prefix %s, asn %s , time %s elem %s" %(pfx, orig, time, elem))
#             print("### Prefix is %s" %pfx)
            asn_time["asn"] = orig
            asn_time["time"] = time
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip

            # Store prefix, origin, timestamp, peer collector in a dictionary in a patricia tree
            pyt_v4.insert(pfx, asn_time)                   
#         print(len(pyt_v4))   
print("Completed")
        

In [4]:
# Program to find the BGP announcements specific for potential scrubbers
# Note: All the helper methods (e.g. detect_ip_version) are at the bottom of the program file
# Step # 1
# Find customers (ASN and prefixes) of DDoS scrubber for a day
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

scrubber_asn = "200020"
stream = pybgpstream.BGPStream(
    from_time="2021-08-03 00:00:01",
    until_time="2021-08-05 23:59:00",
    collectors=["rrc00"],# "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
    filter = "path _"+scrubber_asn+"_" #Look for all the prefixes that contain AS200020 as an immediate provider
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'announcements')


pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]
        orig = as_path.split()[-1]
        second_as = as_path.split()[-2]

        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # 2. Prefix length should be /24 or /48
        # See all the prefixes that has AS200020 as a second last hop and only IPv4
        if second_as == scrubber_asn and detect_ip_version(pfx) == 'IPv4':
#             print("Prefix %s, asn %s , time %s elem %s" %(pfx, orig, time, elem))
#             print("### Prefix is %s" %pfx)
            asn_time["asn"] = orig
            asn_time["time"] = time
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip

            # Store prefix, origin, timestamp, peer collector in a dictionary in a patricia tree
            pyt_v4.insert(pfx, asn_time)                   
#         print(len(pyt_v4))   
print("Completed")
        

Completed


In [5]:
len(pyt_v4)

32

In [6]:
# Store pytricia objects into a csv file
import pandas as pd
data_list = [{'prefix': prefix, 'asn': pyt_v4[prefix]['asn'], 'announced_time': pyt_v4[prefix]['time'], 'peer_asn': pyt_v4[prefix]['peer_asn'], 'peer_ip': pyt_v4[prefix]['peer_ip']} for prefix in pyt_v4]

# transformed_data = [{'prefix': item[0], 'asn': item[1]['asn']} for item in data]


# df = pd.DataFrame(data_list, columns=['Prefix', 'ASN', 'Time', 'peer ASN', 'peer IP'], index=False)
df = pd.DataFrame(data_list)
df.to_csv('/home/shyam/jupy/ddos_scrubber/data/as'+scrubber_asn+'_03_05August_2021_rrc00_pflen_any.csv', index = False)

In [ ]:
# Get number of unique ASNs for each days
df1 = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/as'+scrubber_asn+'_07July_rrc00_pflen_any.csv')
asn_df1 = df1.asn.unique()
asn_df1.sort()
asn_df1

In [ ]:
# Get number of unique prefixes for each days
df1 = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/as200020_01July_rrc00_pflen_any.csv')
prefix_df1 = df1.prefix
prefix_df1

In [ ]:
# See AS path pattern of the potential customer scrubber on the same day for NBIP-NaWaS (AS200020)
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

origin = "12414"
prefix = "213.134.247.0/24" # prefix to check if its less specific announcement include scrubber ASN on its AS path

stream = pybgpstream.BGPStream(
    from_time="2024-07-07 00:00:01 CET",
    until_time="2024-07-07 23:59:00 CET",
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
        filter = "path "+origin+"$" #Look for all the prefixes originated by AS200020
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'announcements')
stream.add_filter('prefix-less', prefix)

pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes
pyt_v4 = pytricia.PyTricia(128) # For storing ipv6 prefixes



# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
#         print("## elem %s" %elem)
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]
        orig = as_path.split()[-1]
        second_as = as_path.split()[-2]

        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # 2. Prefix length should be /24 or /48
        # See all the prefixes that has AS200020 as a second last hop
        if second_as != "200020":# and pfx_len == 24:
            print("Prefix %s, asn %s , time %s, elem %s" %(pfx, orig, time, elem))
            asn_time["asn"] = orig
            asn_time["time"] = time
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip

            # Store prefix, origin, timestamp, peer collector in a dictionary in a patricia tree
            pyt_v4.insert(pfx, asn_time)                   
#         print(len(pyt_v4))   
print("Completed")

In [2]:
# Detects if a given string is an IPv4 or IPv6
import ipaddress

def detect_ip_version(ip_network_str):
    try:
        ip_network = ipaddress.ip_network(ip_network_str, strict=False)
        if isinstance(ip_network, ipaddress.IPv4Network):
            return "IPv4"
        elif isinstance(ip_network, ipaddress.IPv6Network):
            return "IPv6"
    except ValueError:
        return "Invalid IP address or network"

In [ ]:
# Read csv file and find the number of prefixes of different length from /19 to /24
import pandas as pd
import ipaddress

df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/as'+scrubber_asn+'_01July_rrc00_pflen_any.csv')
df['prefix'][0]
ip = ipaddress.ip_network(pfx)
pfx_len = ip.prefixlen
len(df['prefix'])